# Claude Messages API의 새 기능 Message Threads 정리: 대화 전체를 매번 다시 보내던 방식과 무엇이 다른지, thread 파라미터로 긴 에이전트 세션 지연을 줄이는 법

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `ANTHROPIC_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['ANTHROPIC_API_KEY']:
    os.environ[name] = userdata.get(name)

### SDK 설치와 벤치 스크립트 작성

In [ ]:
%%bash
pip install -q -U anthropic httpx matplotlib
mkdir -p lab outputs
cat > lab/bench.py <<'E'
import os, json, time, uuid, httpx, sys
KEY=os.environ["ANTHROPIC_API_KEY"]; URL="https://api.anthropic.com/v1/messages"
MODEL="claude-haiku-4-5-20251001"; BETA="message-threads-2026-08-12"
QS=["이 문서의 핵심 주제를 한 문장으로 말해줘.","첫 번째로 중요한 규칙은 뭐야?","두 번째 규칙은?","방금 말한 규칙 둘을 비교해줘.","예외 상황이 있으면 알려줘.","초보자가 자주 하는 실수는?","체크리스트로 만들어줘.","지금까지 내용을 한 줄로 요약해줘."]
def system(nonce):
    para="규칙 {i}: 서비스 운영 시 로그는 구조화하고, 요청 ID를 모든 계층에 전달하며, 장애 시 롤백 절차를 문서화한다. 참고 코드 {n}."
    return f"문서 ID {nonce}. 짧게(2문장 이내) 한국어로 답하라.\n"+"\n".join(para.format(i=i,n=nonce) for i in range(450))
def call(body, thread=False):
    h={"x-api-key":KEY,"anthropic-version":"2023-06-01","content-type":"application/json"}
    if thread: h["anthropic-beta"]=BETA
    raw=json.dumps(body,ensure_ascii=False).encode()
    t0=time.perf_counter(); ttft=None; mid=None; usage={}; text=""
    with httpx.stream("POST",URL,headers=h,content=raw,timeout=120) as r:
        if r.status_code!=200:
            r.read(); return {"status":r.status_code,"body":r.text,"bytes":len(raw)}
        for line in r.iter_lines():
            if not line.startswith("data:"): continue
            d=json.loads(line[5:])
            if d["type"]=="message_start": mid=d["message"]["id"]; usage.update(d["message"]["usage"])
            elif d["type"]=="content_block_delta" and ttft is None: ttft=time.perf_counter()-t0
            if d["type"]=="content_block_delta" and d["delta"].get("text"): text+=d["delta"]["text"]
            elif d["type"]=="message_delta": usage.update(d["usage"])
    return {"status":200,"id":mid,"ttft":ttft,"bytes":len(raw),"usage":usage,"text":text}
def run(mode,nonce,cache=False):
    sysm=system(nonce); hist=[]; prev=None; rows=[]
    for i,q in enumerate(QS):
        new={"role":"user","content":q}
        extra={"cache_control":{"type":"ephemeral"}} if cache else {}
        if mode=="full":
            hist.append(new); body=dict(model=MODEL,max_tokens=120,system=sysm,messages=list(hist),stream=True,**extra); th=False
        elif i==0:
            hist.append(new); body=dict(model=MODEL,max_tokens=120,system=sysm,messages=list(hist),stream=True,thread={"type":"create"},**extra); th=True
        else:
            body=dict(model=MODEL,max_tokens=120,messages=[new],stream=True,thread={"type":"continue","previous_message_id":prev},**extra); th=True
        r=call(body,th)
        if r["status"]!=200: print("ERR",mode,i,r); sys.exit(1)
        if mode=="full": hist.append({"role":"assistant","content":r["text"]})
        prev=r["id"]; u=r["usage"]
        rows.append(dict(mode=mode,cache=cache,turn=i+1,bytes=r["bytes"],ttft=round(r["ttft"],3),input=u.get("input_tokens"),cc=u.get("cache_creation_input_tokens"),cr=u.get("cache_read_input_tokens"),out=u.get("output_tokens")))
        print(rows[-1],flush=True)
    return rows
if __name__=="__main__":
    allr=[]
    for rep in range(3):
        for mode,cache in [("full",False),("thread",False),("thread",True),("full",True)]:
            allr+=[dict(r,rep=rep) for r in run(mode,uuid.uuid4().hex[:8],cache)]
    json.dump(allr,open("outputs/bench.json","w"),ensure_ascii=False,indent=1)
E

### 전체 재전송·thread·캐싱 4가지 조건 8턴 측정 (G1~G3)

In [ ]:
%%bash
cd lab && python bench.py && cd .. && python - <<'E'
import json,statistics as s,collections
d=json.load(open('outputs/bench.json'))
g=collections.defaultdict(list)
for r in d: g[(r['mode'],r['cache'],r['turn'])].append(r)
for k in sorted(g):
    v=g[k]; print(k,'bytes',v[0]['bytes'],'ttft_med',round(s.median(x['ttft'] for x in v),2),'in',[x['input'] for x in v],'cr',[x['cr'] for x in v])
E

### 전체 재전송과 thread의 요청 크기·TTFT 비교 그래프 만들기

In [ ]:
import json, os, statistics as s, collections
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

os.makedirs('/content/outputs', exist_ok=True)
# 앞 셀에서 만든 측정 결과(bench.json)를 읽는다
path = next(p for p in ['/content/outputs/bench.json', '/content/lab/outputs/bench.json'] if os.path.exists(p))
d = json.load(open(path))

g = collections.defaultdict(list)
for r in d:
    g[(r['mode'], r['cache'], r['turn'])].append(r)

lab = {('full', False): 'full resend', ('thread', False): 'thread', ('thread', True): 'thread+cache', ('full', True): 'full+cache'}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for k, l in lab.items():
    t = range(1, 9)
    ax[0].plot(t, [g[(k[0], k[1], i)][0]['bytes'] for i in t], marker='o', label=l)
    ax[1].plot(t, [s.median(x['ttft'] for x in g[(k[0], k[1], i)]) for i in t], marker='o', label=l)
ax[0].set_yscale('log'); ax[0].set_title('request body bytes (log)')
ax[1].set_title('TTFT median of 3 runs (s)')
for a in ax:
    a.set_xlabel('turn'); a.legend()
plt.tight_layout()
plt.savefig('/content/outputs/compare.png', dpi=130)
display(Image('/content/outputs/compare.png'))

### thread에 cache_control을 켰을 때 cache_read_input_tokens 변화 확인

In [ ]:
import json, os, collections

path = next(p for p in ['/content/outputs/bench.json', '/content/lab/outputs/bench.json'] if os.path.exists(p))
d = json.load(open(path))

g = collections.defaultdict(list)
for r in d:
    g[(r['mode'], r['cache'], r['turn'])].append(r)

# thread 방식에서 캐싱을 껐을 때(False)와 켰을 때(True)의 usage를 턴별로 비교
for k in sorted(g):
    if k[0] != 'thread':
        continue
    v = g[k]
    print(k, 'input', [x['input'] for x in v], 'cc', [x['cc'] for x in v], 'cr', [x['cr'] for x in v])

### 5분 만료 후 raw continue 404와 헬퍼 send() 복구

In [ ]:
%%bash
cd lab && cat > expire.py <<'E'
import os, json, time, httpx2, anthropic
import bench
log=[]
def hook(req):
    b=json.loads(req.content or b'{}')
    log.append((time.strftime('%H:%M:%S'),len(req.content),b.get('thread'),len(b.get('messages',[])),'system' in b))
client=anthropic.Anthropic(http_client=httpx2.Client(event_hooks={'request':[hook]},timeout=120))
M='claude-haiku-4-5-20251001'
th=client.beta.messages.thread(model=M,max_tokens=100,system='짧게 한국어로 답하라.')
r1=th.send({'role':'user','content':'안녕, 내 이름은 민수야.'}); print('helper turn1:',r1.content[0].text)
raw1=bench.call(dict(model=M,max_tokens=100,system='짧게 한국어로 답하라.',messages=[{'role':'user','content':'안녕, 내 이름은 민수야.'}],stream=True,thread={'type':'create'}),True)
time.sleep(330)
raw2=bench.call(dict(model=M,max_tokens=100,messages=[{'role':'user','content':'내 이름이 뭐였지?'}],stream=True,thread={'type':'continue','previous_message_id':raw1['id']}),True)
print('RAW continue after 5.5min:',raw2['status'],raw2.get('body'))
r2=th.send({'role':'user','content':'내 이름이 뭐였지?'}); print('helper turn2:',r2.content[0].text)
for l in log: print(l)
E
python expire.py